In [ ]:
!pip install pytesseract
!apt-get install tesseract-ocr-spa

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tesseract-ocr-spa
0 upgraded, 1 newly installed, 0 to remove and 4 not upgraded.
Need to get 951 kB of archives.
After this operation, 2,309 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-spa all 1:4.00~git30-7274cfa-1.1 [951 kB]
Fetched 951 kB in 1s (726 kB/s)
Selecting previously unselected package tesseract-ocr-spa.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-spa_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-spa (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-spa (1:4.00~git30-7274cfa-1.1) ...


Después de cargar las herramientas de lectura, se utilizan las siguientes librerías. 

In [ ]:
import pandas as pd
from PIL import Image, ImageOps
import pytesseract
import json
from os import listdir
import re

Se crea una lista de archivos. Es importante que sigan el orden, non-par

In [ ]:
files= listdir()
files = sorted([item for item in files if "JPEG" in item])

Este es el código de segmentación. Está compuesto de cuatro partes:
1. Definición de ratios óptimos. Esto es a discreción del usuario. 
2. Uso de loop en lista de archivos. Dentro de ésta. 
2.1 Se abre la imagen y se transpone a vertical. 
2.2 Se obtienen las medidas de la imange. 
2.3 Se definen las coordenadas de segmentación óptimas de acuerdo a si se trata de página non o par, la cual se define simplemente por el orden de los archivos. 
2.4 Se segmenta la imagen. 
2.5 Se lee la imagen de cada segmento. 
2.6 Se extrae información para las dos versiones: text_data para la versión en txt y text para la versión csv (para esta iteración, la versión en txt es más efectiva)
3. Se crea el archivo data.txt Versión 1
4. Se comienza la limpieza y se crea archivo csv. Versión 2

In [ ]:
text = "" #1
oddhorratio=0.49
evenhorratio=0.51
verratio=0.085
data=[]
for i in range(0,len(files)): #2
    #2.1
    nombre = files[i] 
    img = Image.open(nombre)
    img = ImageOps.exif_transpose(img)
    ancho = img.size[0]
    alto = img.size[1]
    #2.2
    if i%2==1 :#2.2
      img_izq_area = (0, alto*verratio, ancho*evenhorratio, alto)
      img_der_area = (ancho*evenhorratio, alto*verratio, ancho, alto)
    else :
      img_izq_area = (0, alto*verratio, ancho*oddhorratio, alto)
      img_der_area = (ancho*oddhorratio, alto*verratio, ancho, alto)
    #2.3
    img_izq = img.crop(img_izq_area)
    img_der = img.crop(img_der_area)
    #2.4
    text_izq = pytesseract.image_to_string(img_izq, lang='spa')
    text_der = pytesseract.image_to_string(img_der, lang='spa')

    #2.5
    text_data = {
        "image": nombre,
        "left": text_izq,
        "right": text_der
    }
    texto = text_izq + "\n" + text_der
    text = text + "\n" + texto
    data.append(text_data)

#3
with open("data.txt", "w", encoding="utf-8") as fp:
  for item in data:
    fp.write(str(item) + "\n")

#4
textosep= text.split("\n\n")
textosep = [re.sub(r"[^a-zA-Z0-9\s.,!?;:'\"()\-]", "", item).strip() for item in textosep]
textosep = [re.sub(r"^\n*", "", item).strip() for item in textosep]
textosep = [re.sub(r"[\x00-\x1f\x7f]+", " ", item).strip() for item in textosep]
type = []
patron = re.compile(r"^[A-Z]{2}.*[A-Z]{2}$|[A-Z]{5}$|^[A-Z]{5}")
for item in textosep:
  if patron.search(item):
    tipo="nombre"
  else:
    tipo="otro"
  type.append(tipo)


d = {"text": textosep, "tipo": type}
df = pd.DataFrame(data=d)
df.to_csv("texto_imagenes.csv", index=False, encoding="utf-8-sig")